# Banking77 Intent Classification with BERT

**Section 2 — Project Framework, Data Loading & Exploratory Analysis (EDA only)**

Scope of this section: CRISP-DM framing, dataset verification, and EDA/visualization.
No model training here — no MLP baseline, no RoBERTa/BERT fine-tuning (those belong to later sections).

## 1. Project Framework — CRISP-DM applied to Banking77

Dataset: Banking77 — 10,003 train / 3,080 test rows, columns `text, category`, 77 customer-service intents.

| # | CRISP-DM phase | How it applies to this project |
|---|---|---|
| 1 | **Business Understanding** | Automate customer-query routing for a bank's support chatbot. Success = high **first-response accuracy** (correct intent on first try) and reduced **triage time** (faster handoff to the right team/flow). Baseline expectation: a majority-class guess would score ~1/77, so any useful model must clearly beat that. |
| 2 | **Data Understanding** | This section (Section 2): verify row counts, schema, 77 intents in each split, missing/duplicates, intent distribution, text-length patterns, and example queries. |
| 3 | **Data Preparation** | Later section: text cleaning (light — keep intent signal), stratified train/validation split, tokenizer encoding, padding/truncation, class-imbalance handling if needed. |
| 4 | **Modeling** | Later sections: classical/MLP baseline first, then fine-tuned transformer (BERT/RoBERTa) for intent classification. |
| 5 | **Evaluation** | Later section: accuracy + macro-F1 (77 classes, imbalance-sensitive), per-intent precision/recall, confusion-matrix review of confusable intents. |
| 6 | **Deployment** | Later section: export model + tokenizer, inference pipeline with confidence threshold and fallback-to-human, monitoring for intent drift. **PII note:** this is a regulated industry (banking) — raw queries may contain emails, phones, or card numbers, so the pipeline must mask/hash PII before logging or training (see PII stub at the end of this notebook). |

## 2. Load and Verify Data

In [ ]:
import pandas as pd
import numpy as np

TRAIN_PATH = "datasets/banking77_train.csv"
TEST_PATH = "datasets/banking77_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# --- Shape and schema assertions ---
assert train_df.shape[0] == 10003, f"Unexpected train rows: {train_df.shape[0]}"
assert test_df.shape[0] == 3080, f"Unexpected test rows: {test_df.shape[0]}"
assert list(train_df.columns) == ["text", "category"], train_df.columns.tolist()
assert list(test_df.columns) == ["text", "category"], test_df.columns.tolist()
assert train_df["category"].nunique() == 77, train_df["category"].nunique()
assert test_df["category"].nunique() == 77, test_df["category"].nunique()

print(f"train: {train_df.shape} | test: {test_df.shape}")
print(f"train intents: {train_df['category'].nunique()} | test intents: {test_df['category'].nunique()}")
print(f"intent sets identical: {set(train_df['category'].unique()) == set(test_df['category'].unique())}")

# --- Data quality: missing values and exact duplicates ---
print("\nMissing values (train):", train_df.isnull().sum().to_dict())
print("Missing values (test):", test_df.isnull().sum().to_dict())
print(f"Exact-duplicate rows — train: {train_df.duplicated().sum()}, test: {test_df.duplicated().sum()}")

# --- Example rows ---
print("\n--- train head ---")
display(train_df.head(5))
print("--- test head ---")
display(test_df.head(5))

## 3. Explore and Visualize

Goals: intent distribution (which intents dominate?), text-length patterns overall and per intent,
representative examples, imbalance assessment, and a PII-handling stub for the regulated banking context.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

intent_counts = train_df["category"].value_counts()
print(f"Intents: {len(intent_counts)} | min count: {intent_counts.min()} | max count: {intent_counts.max()}")
print(f"Mean per intent: {intent_counts.mean():.1f} | std: {intent_counts.std():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top10 = intent_counts.head(10)
sns.barplot(x=top10.values, y=top10.index, ax=axes[0], color="steelblue")
axes[0].set_title("Top 10 intents (train)")
axes[0].set_xlabel("Count")

bottom10 = intent_counts.tail(10).sort_values(ascending=True)
sns.barplot(x=bottom10.values, y=bottom10.index, ax=axes[1], color="coral")
axes[1].set_title("Bottom 10 intents (train)")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

print("\nTop 10:", top10.to_dict())
print("Bottom 10:", bottom10.sort_values().to_dict())

In [ ]:
# Text-length features (characters and words)
train_df["n_chars"] = train_df["text"].str.len()
train_df["n_words"] = train_df["text"].str.split().str.len()

print("--- Overall text-length stats (train) ---")
display(train_df[["n_chars", "n_words"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_df["n_chars"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of query length (characters)")
axes[0].set_xlabel("Characters")

sns.boxplot(x=train_df["n_words"], ax=axes[1], color="lightblue")
axes[1].set_title("Spread of query length (words)")
axes[1].set_xlabel("Words")
plt.tight_layout()
plt.show()

# Per-intent length: which intents run long vs short?
per_intent_len = train_df.groupby("category")["n_words"].agg(["mean", "median", "count"]).round(2)
per_intent_len = per_intent_len.sort_values("mean", ascending=False)
print("--- Longest 5 intents by mean word count ---")
display(per_intent_len.head(5))
print("--- Shortest 5 intents by mean word count ---")
display(per_intent_len.tail(5))

# Boxplot for a readable subset (top-5 most frequent intents)
top5_intents = intent_counts.head(5).index.tolist()
subset = train_df[train_df["category"].isin(top5_intents)]
plt.figure(figsize=(12, 4))
sns.boxplot(data=subset, x="n_words", y="category", order=top5_intents)
plt.title("Word-count spread for top-5 intents")
plt.xlabel("Words")
plt.tight_layout()
plt.show()

In [ ]:
# Representative examples: 2-3 queries for a few sample intents
np.random.seed(42)
sample_intents = ["card_arrival", "lost_or_stolen_card", "exchange_rate", "fiat_currency_support"]
sample_intents = [c for c in sample_intents if c in train_df["category"].unique()]

for intent in sample_intents:
    print(f"=== {intent} ===")
    for q in train_df.loc[train_df['category'] == intent, 'text'].sample(3, random_state=42):
        print(f"  - {q}")
    print()

# --- Class imbalance note ---
ratio = intent_counts.max() / intent_counts.min()
print(f"Imbalance ratio (max/min per-intent count): {ratio:.2f}")
print(
    "Note: Banking77 is roughly balanced by design (each intent has 100+ train examples), "
    "so no resampling is applied in EDA. If the confusion matrix (later section) shows weak recall "
    "on rare/confusable intents, consider stratified splits, class weights, or focal loss during modeling."
)

### Word-level analysis — which words carry the intent signal?

Bag-of-words view (train split only, `CountVectorizer` with English stop words removed and a bounded
vocabulary): most common words overall, per-intent keywords for the top-6 intents, overlap between
intents, and time/urgency language. Lightweight EDA only — no modeling here.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe: no GUI backend required
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

TOP_N_OVERALL = 20
TOP_N_PER_INTENT = 10
N_INTENTS_WORDS = 6
MAX_FEATURES = 2000  # bounded vocabulary keeps cells light

top_intents_words = intent_counts.head(N_INTENTS_WORDS).index.tolist()
print(f"Intents analyzed for word patterns: {top_intents_words}")

# --- Overall top words (English stop words removed) ---
overall_vec = CountVectorizer(stop_words="english", max_features=MAX_FEATURES)
X_all = overall_vec.fit_transform(train_df["text"])
overall_freq = np.asarray(X_all.sum(axis=0)).ravel()
overall_top_idx = overall_freq.argsort()[::-1][:TOP_N_OVERALL]
overall_words = [overall_vec.get_feature_names_out()[i] for i in overall_top_idx]
overall_counts = overall_freq[overall_top_idx]

plt.figure(figsize=(10, 6))
sns.barplot(x=overall_counts, y=overall_words, color="steelblue")
plt.title(f"Top {TOP_N_OVERALL} words overall (train, stop words removed)")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

print("Top words overall:", dict(zip(overall_words, overall_counts.tolist())))

# --- Per-intent top words for the top-6 intents ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, intent in zip(axes.ravel(), top_intents_words):
    vec = CountVectorizer(stop_words="english", max_features=MAX_FEATURES)
    X = vec.fit_transform(train_df.loc[train_df["category"] == intent, "text"])
    freq = np.asarray(X.sum(axis=0)).ravel()
    top_idx = freq.argsort()[::-1][:TOP_N_PER_INTENT]
    words = [vec.get_feature_names_out()[i] for i in top_idx]
    counts = freq[top_idx]
    sns.barplot(x=counts, y=words, ax=ax, color="steelblue")
    ax.set_title(intent)
    ax.set_xlabel("Count")
plt.suptitle(f"Top {TOP_N_PER_INTENT} words per top-{N_INTENTS_WORDS} intents", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Word overlap across the top-6 intents (shared top-words matrix)
TOP_K_OVERLAP = 15
top_word_sets = {}
for intent in top_intents_words:
    vec = CountVectorizer(stop_words="english", max_features=MAX_FEATURES)
    X = vec.fit_transform(train_df.loc[train_df["category"] == intent, "text"])
    freq = np.asarray(X.sum(axis=0)).ravel()
    top_idx = freq.argsort()[::-1][:TOP_K_OVERLAP]
    top_word_sets[intent] = set(vec.get_feature_names_out()[i] for i in top_idx)

overlap = pd.DataFrame(
    [[len(top_word_sets[a] & top_word_sets[b]) for b in top_intents_words] for a in top_intents_words],
    index=top_intents_words,
    columns=top_intents_words,
)
print(f"--- Pairwise shared top-{TOP_K_OVERLAP} words ---")
display(overlap)

plt.figure(figsize=(8, 6))
sns.heatmap(overlap, annot=True, fmt="d", cmap="Blues")
plt.title(f"Shared top-{TOP_K_OVERLAP} words between intents")
plt.tight_layout()
plt.show()

from collections import Counter
shared_counter = Counter()
for words in top_word_sets.values():
    shared_counter.update(words)
recurring = sorted([w for w, c in shared_counter.items() if c >= 3])
print(f"Words in top-{TOP_K_OVERLAP} of 3+ intents ({len(recurring)}): {recurring}")
print(
    "Note: generic words recurring across intents carry little signal on their own — "
    "intent hinges on the words around them."
)

# --- Focused comparison of two similar card-related intents ---
preferred_pair = ("card_arrival", "card_linking")
if all(p in train_df["category"].unique() for p in preferred_pair):
    intent_a, intent_b = preferred_pair
else:  # fallback: most overlapping pair among the top-6
    best, best_val = None, -1
    for i in range(len(top_intents_words)):
        for j in range(i + 1, len(top_intents_words)):
            v = overlap.iloc[i, j]
            if v > best_val:
                best, best_val = (top_intents_words[i], top_intents_words[j]), v
    intent_a, intent_b = best
print(f"\n--- Focused comparison: {intent_a} vs {intent_b} ---")

pair_mask = train_df["category"].isin([intent_a, intent_b])
pair_vec = CountVectorizer(stop_words="english", max_features=MAX_FEATURES)
X_pair = pair_vec.fit_transform(train_df.loc[pair_mask, "text"]).toarray()
labels_pair = train_df.loc[pair_mask, "category"].to_numpy()
mean_a = X_pair[labels_pair == intent_a].mean(axis=0)
mean_b = X_pair[labels_pair == intent_b].mean(axis=0)
vocab = np.array(pair_vec.get_feature_names_out())
diff = mean_a - mean_b  # positive -> distinctive of A, negative -> distinctive of B
top_a_idx = diff.argsort()[::-1][:8]
top_b_idx = diff.argsort()[:8]
print(f"Distinctive of {intent_a}: {vocab[top_a_idx].tolist()}")
print(f"Distinctive of {intent_b}: {vocab[top_b_idx].tolist()}")

sel_idx = np.concatenate([top_a_idx, top_b_idx])
sel_diff = diff[sel_idx]
order = sel_diff.argsort()
colors = ["coral" if v < 0 else "steelblue" for v in sel_diff[order]]
plt.figure(figsize=(10, 6))
plt.barh(np.arange(len(order)), sel_diff[order], color=colors)
plt.yticks(np.arange(len(order)), vocab[sel_idx][order])
plt.axvline(0, color="black", linewidth=0.8)
plt.title(f"Differentiating keywords: {intent_a} (right) vs {intent_b} (left)")
plt.xlabel("Mean frequency difference")
plt.tight_layout()
plt.show()

In [ ]:
# Temporal / urgency language check (regex word-boundary match, case-insensitive via lowercasing)
import re

TIME_WORDS = ["today", "tomorrow", "week", "month", "urgent", "asap",
              "soon", "immediately", "now", "delay", "late", "quick", "fast"]

text_lower = train_df["text"].str.lower()
time_hits = pd.DataFrame(
    {w: text_lower.str.contains(rf"\b{re.escape(w)}\b", regex=True) for w in TIME_WORDS}
)
overall_time_rate = time_hits.any(axis=1).mean()
print(f"Queries containing any time/urgency word: {overall_time_rate:.1%}")
per_word_rate = (time_hits.mean().sort_values(ascending=False) * 100).round(2)
print("--- Per-word hit rate (% of train queries) ---")
display(per_word_rate.to_frame("pct_queries"))

plt.figure(figsize=(10, 4))
sns.barplot(x=per_word_rate.values, y=per_word_rate.index, color="steelblue")
plt.title("Time/urgency word frequency (train)")
plt.xlabel("% of queries")
plt.tight_layout()
plt.show()

# Does temporal language vary by intent?
train_time = train_df[["category"]].copy()
train_time["has_time_word"] = time_hits.any(axis=1).to_numpy()
by_intent_time = (train_time.groupby("category")["has_time_word"].mean() * 100).round(2)
top_time = by_intent_time.loc[top_intents_words].sort_values(ascending=False)
print("--- Time-word rate for top-6 intents (% of queries) ---")
display(top_time.to_frame("pct_with_time_word"))

plt.figure(figsize=(10, 4))
sns.barplot(x=top_time.values, y=top_time.index, color="steelblue")
plt.title("Time-word usage varies by intent (top-6)")
plt.xlabel("% of queries with a time/urgency word")
plt.tight_layout()
plt.show()
print(f"Range across top-6 intents: {top_time.min():.1f}% – {top_time.max():.1f}%")

**Initial insights.** Vocabulary overlaps heavily across intents (generic words recur in several top-word lists), so intent labels should be encoded numerically with a saved mapping and stratified splits rather than inferred from keywords; for an MLP baseline this points to TF-IDF preprocessing with English stop-words removed and a capped vocabulary plus attention to confusable pairs, while for transformer fine-tuning it favors subword tokenization with truncation/padding so word order and urgency cues are preserved. No modeling is done here — these directions carry into the preparation and modeling sections.

### PII protection strategy (stub — no real data mutation in EDA)

Banking queries can contain emails, phone numbers, or card numbers. Before any logging,
training export, or deployment demo, PII must be masked or hashed. The stub below defines
detection patterns and a hashing helper, demonstrated on **synthetic strings only** —
the real `train_df`/`test_df` are left untouched in this section.

In [ ]:
import re
import hashlib

# Detection patterns (stub — extend with IBAN/SSN rules per compliance review)
PII_PATTERNS = {
    "email": re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", re.IGNORECASE),
    "phone": re.compile(r"\+?\d[\d\s().-]{7,}\d"),
    "card": re.compile(r"\b(?:\d[ -]?){13,19}\b"),  # 13-19 digit sequences (candidate card numbers)
}

def hash_pii(value: str, salt: str = "banking77") -> str:
    """One-way hash for a detected PII value (keeps joinability without storing raw PII)."""
    return hashlib.sha256(f"{salt}:{value}".encode()).hexdigest()[:16]

def find_pii(text: str) -> dict:
    """Return detected PII spans per category (detection only — no mutation)."""
    return {name: pat.findall(text) for name, pat in PII_PATTERNS.items()}

# Demo on synthetic strings only — never on real dataset rows in EDA
demo_texts = [
    "Contact me at jane.doe@example.com about my card",
    "Call me back on +1 555-123-4567 regarding the transfer",
    "My card 4111 1111 1111 1111 is not working",
]
for t in demo_texts:
    print(f"TEXT: {t}")
    print(f"  detected: {find_pii(t)}")
    print(f"  hashed email demo: {hash_pii('jane.doe@example.com')}")
    break  # one hashing example is enough; detection shown for all below
for t in demo_texts[1:]:
    print(f"TEXT: {t}")
    print(f"  detected: {find_pii(t)}")

## Section 2 — Wrap-up

- Data verified: 10,003 train / 3,080 test rows, `text, category` schema, 77 intents per split, no missing values or exact duplicates.
- EDA shows the intent distribution shape, query-length patterns, and representative examples above.
- PII handling is stubbed (detect + hash) and will be enforced before any training/logging step.
- Next (later sections): data preparation, MLP baseline, transformer fine-tuning, evaluation, deployment.

## 4. MLP Baseline — Preprocessing (TF-IDF + Label Encoding + PyTorch Loaders)

Scope: classical-baseline preparation ONLY — encode intent labels, vectorize queries with TF-IDF,
and wrap them in PyTorch `Dataset`/`DataLoader` objects. No classifier is trained here;
model definition, training, and evaluation belong to the next cells/sections.

In [ ]:
# 4.1 Encode intent labels with LabelEncoder (fit on train, apply to train + test)
from sklearn.preprocessing import LabelEncoder
import pandas as pd

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["category"])
y_test = label_encoder.transform(test_df["category"])

num_classes = len(label_encoder.classes_)
assert num_classes == 77, f"Expected 77 intents, got {num_classes}"
assert set(y_train) == set(range(num_classes))
assert y_train.shape[0] == len(train_df) and y_test.shape[0] == len(test_df)

print(f"num classes: {num_classes}")
print(f"y_train shape: {y_train.shape} | y_test shape: {y_test.shape}")

# Mapping table sample (label <-> intent)
mapping_df = pd.DataFrame({"label": range(num_classes), "intent": label_encoder.classes_})
display(mapping_df.head(10))
print("NOTE: persist this fitted LabelEncoder mapping (e.g. pickle classes_) for evaluation —")
print("test labels and all later predictions must reuse it so ids map back to the same intents.")

In [ ]:
# 4.2 Vectorize query texts with TF-IDF (fit on train ONLY)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
X_train_tfidf = tfidf.fit_transform(train_df["text"])
X_test_tfidf = tfidf.transform(test_df["text"])

vocab_size = len(tfidf.get_feature_names_out())
assert X_train_tfidf.shape[0] == len(train_df)
assert X_test_tfidf.shape[0] == len(test_df)
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1] == vocab_size

print(f"vocab size: {vocab_size}")
print(f"train TF-IDF matrix: {X_train_tfidf.shape}")
print(f"test TF-IDF matrix:  {X_test_tfidf.shape}")
print(
    "No leakage: the vectorizer is FIT on train queries only (vocabulary + idf from train), "
    "then applied via transform() to test — test wording never influences the vocabulary or weights."
)

In [ ]:
# 4.3 PyTorch Dataset + DataLoaders over TF-IDF features (batching only, no training)
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TfidfIntentDataset(Dataset):
    """Dense-tensor wrapper around a TF-IDF sparse matrix + integer labels."""
    def __init__(self, X_sparse, y_array):
        self.X = torch.from_numpy(X_sparse.toarray().astype(np.float32))
        self.y = torch.from_numpy(np.asarray(y_array, dtype=np.int64))
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TfidfIntentDataset(X_train_tfidf, y_train)
test_dataset = TfidfIntentDataset(X_test_tfidf, y_test)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

assert len(train_dataset) == len(train_df) == X_train_tfidf.shape[0]
assert len(test_dataset) == len(test_df) == X_test_tfidf.shape[0]

print(f"train batches: {len(train_loader)} | test batches: {len(test_loader)} (batch={BATCH_SIZE})")

# Show one batch shapes (features + integer labels)
X_batch, y_batch = next(iter(train_loader))
print(f"one train batch — X: {tuple(X_batch.shape)} ({X_batch.dtype}), y: {tuple(y_batch.shape)} ({y_batch.dtype})")
assert X_batch.shape[1] == vocab_size
assert y_batch.min() >= 0 and y_batch.max() < num_classes

## 5. MLP Baseline — Training and Evaluation

Scope: define a PyTorch MLP on the TF-IDF features from Section 4, train it with cross-entropy + AdamW, and evaluate on the held-out test split (accuracy, macro/weighted F1, per-intent report, confusable pairs). This is the classical baseline; transformer fine-tuning belongs to a later section.

In [ ]:
# 5.1 MLP classifier: input dim comes from the fitted TF-IDF vocabulary at runtime
import random
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class IntentMLP(nn.Module):
    """TF-IDF -> 512 (ReLU+Dropout) -> 256 (ReLU+Dropout) -> 77 intent logits."""
    def __init__(self, input_dim: int, num_classes: int, dropout_p: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.net(x)

# NOTE: vocab_size is read from the fitted vectorizer — never hardcoded.
print(f"runtime vocab_size: {vocab_size} | num_classes: {num_classes}")
mlp = IntentMLP(input_dim=vocab_size, num_classes=num_classes)
print(mlp)
n_params = sum(p.numel() for p in mlp.parameters())
print(f"trainable parameters: {n_params:,}")

# Sanity check: one batch -> 77 logits
mlp.eval()
with torch.no_grad():
    Xb, yb = next(iter(train_loader))
    logits = mlp(Xb)
    assert logits.shape == (Xb.shape[0], num_classes), logits.shape
    print(f"sanity forward — input: {tuple(Xb.shape)} -> logits: {tuple(logits.shape)}")


In [ ]:
# 5.2 Train with CrossEntropyLoss + AdamW; report avg loss and accuracy per epoch
import matplotlib.pyplot as plt

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 12  # small on purpose: ~10k x 2k dense inputs train in minutes on CPU

history = {"loss": [], "acc": []}
mlp.train()
for epoch in range(1, EPOCHS + 1):
    running_loss, correct, total = 0.0, 0, 0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        logits = mlp(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * Xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += Xb.size(0)
    avg_loss, acc = running_loss / total, correct / total
    history["loss"].append(avg_loss)
    history["acc"].append(acc)
    print(f"epoch {epoch:2d}/{EPOCHS} — loss: {avg_loss:.4f} | train acc: {acc:.4f}")

# Loss / accuracy curve
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(range(1, EPOCHS + 1), history["loss"], marker="o", label="train loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Avg cross-entropy loss")
ax2 = ax1.twinx()
ax2.plot(range(1, EPOCHS + 1), history["acc"], marker="s", color="coral", label="train acc")
ax2.set_ylabel("Train accuracy")
plt.title("MLP training curve (TF-IDF baseline)")
fig.tight_layout()
plt.show()


In [ ]:
# 5.3 Evaluate on test_loader: accuracy, macro/weighted F1, per-intent report, confusable pairs
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

mlp.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        all_preds += mlp(Xb).argmax(dim=1).tolist()
        all_true += yb.tolist()

test_acc = accuracy_score(all_true, all_preds)
macro_f1 = f1_score(all_true, all_preds, average="macro")
weighted_f1 = f1_score(all_true, all_preds, average="weighted")
print(f"test accuracy: {test_acc:.4f} | macro F1: {macro_f1:.4f} | weighted F1: {weighted_f1:.4f}")

report = classification_report(
    all_true, all_preds, target_names=list(label_encoder.classes_), digits=3, output_dict=True
)
per_intent = pd.DataFrame(report).T.iloc[:-3]  # drop accuracy/macro/weighted rows
print(f"\nmacro avg: P={report['macro avg']['precision']:.3f} "
      f"R={report['macro avg']['recall']:.3f} F1={report['macro avg']['f1-score']:.3f}")
print(f"weighted avg: P={report['weighted avg']['precision']:.3f} "
      f"R={report['weighted avg']['recall']:.3f} F1={report['weighted avg']['f1-score']:.3f}")
print("\n--- Top-5 intents by F1 ---")
display(per_intent.sort_values("f1-score", ascending=False).head(5).round(3))
print("--- Bottom-5 intents by F1 ---")
display(per_intent.sort_values("f1-score").head(5).round(3))

# Confusable pairs: largest off-diagonal cells of the confusion matrix
cm = confusion_matrix(all_true, all_preds, labels=list(range(num_classes)))
names = list(label_encoder.classes_)
pairs = [(cm[i, j], names[i], names[j])
         for i in range(num_classes) for j in range(num_classes) if i != j]
pairs.sort(reverse=True)
print("\n--- Top-5 confusable pairs (true -> predicted: count) ---")
for count, true_name, pred_name in pairs[:5]:
    print(f"  {true_name} -> {pred_name}: {count}")


## 6. RoBERTa Transformer — Preprocessing (Tokenizer + PyTorch Loaders)

Scope: transformer-input preparation ONLY — load the `roberta-base` tokenizer, encode queries to `input_ids` + `attention_mask`, reuse the MLP section's label mapping, and wrap everything in PyTorch `Dataset`/`DataLoader` objects. No model weights, no LoRA, no fine-tuning here; training belongs to the next section.

In [ ]:
# 6.1 Load the roberta-base tokenizer (tokenizer files only — no model weights needed yet)
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"
try:
    roberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"loaded tokenizer: {MODEL_NAME}")
except OSError as e:
    raise RuntimeError(
        f"Could not download '{MODEL_NAME}' (offline?). "
        "Check network access or set HF_HUB_OFFLINE=1 with a cached copy. "
        f"Original error: {e}"
    ) from e

print(f"vocab size: {roberta_tokenizer.vocab_size}")
print(f"special tokens: BOS={roberta_tokenizer.bos_token} SEP={roberta_tokenizer.sep_token} "
      f"PAD={roberta_tokenizer.pad_token} (id={roberta_tokenizer.pad_token_id}) "
      f"MASK={roberta_tokenizer.mask_token}")
print(f"model max length: {roberta_tokenizer.model_max_length}")


In [ ]:
# 6.2 Tokenize train/test queries with truncation + padding (max_length 128)
MAX_LEN = 128

train_enc = roberta_tokenizer(
    train_df["text"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN
)
test_enc = roberta_tokenizer(
    test_df["text"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN
)

import numpy as np

train_input_ids = np.asarray(train_enc["input_ids"], dtype=np.int64)
train_attention = np.asarray(train_enc["attention_mask"], dtype=np.int64)
test_input_ids = np.asarray(test_enc["input_ids"], dtype=np.int64)
test_attention = np.asarray(test_enc["attention_mask"], dtype=np.int64)

assert train_input_ids.shape == (len(train_df), MAX_LEN)
assert train_attention.shape == (len(train_df), MAX_LEN)
assert test_input_ids.shape == (len(test_df), MAX_LEN)
assert test_attention.shape == (len(test_df), MAX_LEN)

print(f"train input_ids:     {train_input_ids.shape}")
print(f"train attention_mask: {train_attention.shape}")
print(f"test input_ids:      {test_input_ids.shape}")
print(f"test attention_mask: {test_attention.shape}")
print(f"tokenizer vocab size: {roberta_tokenizer.vocab_size}")

# Sample decode of the first train query (ids -> subwords -> text)
sample_text = train_df["text"].iloc[0]
sample_ids = train_input_ids[0]
sample_tokens = roberta_tokenizer.convert_ids_to_tokens(sample_ids[:16])
print(f"\noriginal query: {sample_text!r}")
print(f"first 16 token ids: {sample_ids[:16].tolist()}")
print(f"first 16 tokens: {sample_tokens}")
print(f"decoded back: {roberta_tokenizer.decode(sample_ids, skip_special_tokens=True)!r}")


In [ ]:
# 6.3 Reuse the SAME LabelEncoder mapping from the MLP section (do NOT refit)
# Refitting would reshuffle ids 0-76 and silently break comparability with the MLP baseline.
assert "label_encoder" in globals(), "label_encoder missing — run Section 4 (MLP preprocessing) first"
assert len(label_encoder.classes_) == 77, f"Expected 77 classes, got {len(label_encoder.classes_)}"

# Re-apply (transform only) so the RoBERTa tensors share the identical id mapping
y_train_ids = label_encoder.transform(train_df["category"])
y_test_ids = label_encoder.transform(test_df["category"])
assert set(y_train_ids) == set(range(77)) and set(y_test_ids) <= set(range(77))
assert (y_train_ids == y_train).all() and (y_test_ids == y_test).all(), \
    "RoBERTa ids diverged from the MLP section mapping — check label_encoder state"

print(f"classes: {len(label_encoder.classes_)} (matches MLP section)")
print(f"y_train_ids: {y_train_ids.shape} | y_test_ids: {y_test_ids.shape}")
print(f"first 5 train labels: {y_train_ids[:5].tolist()} -> "
      f"{[label_encoder.classes_[i] for i in y_train_ids[:5]]}")
print("Mapping VERIFIED identical to Section 4 — same LabelEncoder object, transform() only.")


In [ ]:
# 6.4 PyTorch Dataset + DataLoaders over (input_ids, attention_mask, labels)
import torch
from torch.utils.data import Dataset, DataLoader

class RobertaIntentDataset(Dataset):
    """Integer token ids + attention mask + integer intent label per query."""
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = torch.from_numpy(input_ids).long()
        self.attention_mask = torch.from_numpy(attention_mask).long()
        self.labels = torch.from_numpy(np.asarray(labels, dtype=np.int64))
    def __len__(self):
        return self.input_ids.shape[0]
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

train_roberta = RobertaIntentDataset(train_input_ids, train_attention, y_train_ids)
test_roberta = RobertaIntentDataset(test_input_ids, test_attention, y_test_ids)

ROBERTA_BATCH = 32
train_roberta_loader = DataLoader(train_roberta, batch_size=ROBERTA_BATCH, shuffle=True)
test_roberta_loader = DataLoader(test_roberta, batch_size=ROBERTA_BATCH, shuffle=False)

assert len(train_roberta) == len(train_df) and len(test_roberta) == len(test_df)
print(f"train batches: {len(train_roberta_loader)} | test batches: {len(test_roberta_loader)} "
      f"(batch={ROBERTA_BATCH})")

# Batch demo: shapes of one batch
batch = next(iter(train_roberta_loader))
print(f"one train batch — input_ids: {tuple(batch['input_ids'].shape)} | "
      f"attention_mask: {tuple(batch['attention_mask'].shape)} | "
      f"labels: {tuple(batch['labels'].shape)}")
assert batch["input_ids"].shape[1] == MAX_LEN
assert batch["labels"].min() >= 0 and batch["labels"].max() < 77


**TF-IDF vs subword tokenization.** TF-IDF (Section 4) counts whole words: each query becomes a sparse vector over a fixed word vocabulary, so word order is discarded and unseen words vanish (out-of-vocabulary). RoBERTa subword tokenization instead splits words into frequent subword pieces, preserves word order in a fixed-length `input_ids` sequence with an `attention_mask`, and keeps context so the model can distinguish word senses and urgency cues from surrounding words. Tradeoff: TF-IDF is cheap and order-free; subwords cost more compute but carry the order and context signal the transformer fine-tuning step needs.

## 7. RoBERTa + LoRA — Fine-tuning and Evaluation (RUN ON COLAB GPU)

Scope: parameter-efficient fine-tuning of `roberta-base` for the 77 banking intents with LoRA, then test-set evaluation in the SAME format as the MLP baseline (Section 5) for direct comparison.

> **RUN ON COLAB GPU (T4/L4) — do NOT run full training on a CPU-only box** (it takes hours). All cells in this section expect the Section 6 objects (`roberta_tokenizer`, `train_roberta`, `test_roberta`, `test_roberta_loader`, `label_encoder`, `MODEL_NAME`) to already exist in the session.
>
> Expected outcome: the LoRA-tuned RoBERTa clearly beats the MLP baseline (test acc 0.847) on accuracy and macro-F1, with the largest gains on the confusable pairs flagged in Section 5.

In [ ]:
# 7.0 Colab GPU setup — RUN ON COLAB GPU (skip full training on CPU-only machines)
# On Colab, uncomment and run once first:
# !pip install -q -U "transformers[torch]" peft accelerate scikit-learn

import random
import numpy as np
import torch

SEED_ROBERTA = 42
random.seed(SEED_ROBERTA)
np.random.seed(SEED_ROBERTA)
torch.manual_seed(SEED_ROBERTA)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device} | cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected — run this section on Colab GPU (T4/L4). CPU fine-tuning takes hours.")

# Section 6 prerequisites (fail fast with a clear message if the session was restarted)
for _name in ["roberta_tokenizer", "train_roberta", "test_roberta", "test_roberta_loader", "label_encoder", "MODEL_NAME"]:
    assert _name in globals(), f"{_name} missing — run Section 6 (RoBERTa preprocessing) first"
assert len(label_encoder.classes_) == 77, f"Expected 77 intents, got {len(label_encoder.classes_)}"
print(f"prerequisites OK — backbone: {MODEL_NAME} | train: {len(train_roberta)} | test: {len(test_roberta)} | intents: 77")


In [ ]:
# 7.1 Fine-tuning config — RUN ON COLAB GPU
# Conservative single-GPU recipe for 77-way intent classification (matches the MLP comparison setup).
from transformers import TrainingArguments

roberta_args = TrainingArguments(
    output_dir="roberta-banking77-lora",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    seed=42,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

print(f"lr: {roberta_args.learning_rate} | epochs: {roberta_args.num_train_epochs} | "
      f"train batch: {roberta_args.per_device_train_batch_size} | eval batch: {roberta_args.per_device_eval_batch_size} | "
      f"seed: {roberta_args.seed} | fp16: {roberta_args.fp16}")


In [ ]:
# 7.2 LoRA-wrapped roberta-base — RUN ON COLAB GPU
# Trains <1% of parameters: frozen RoBERTa backbone + low-rank adapters on query/key/value projections.
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

NUM_LABELS = 77  # must match the 77 Banking77 intents from Sections 4/6
assert len(label_encoder.classes_) == NUM_LABELS

base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value"],
    bias="none",
)
print(f"LoRA: rank={lora_config.r} alpha={lora_config.lora_alpha} dropout={lora_config.lora_dropout} "
      f"targets={lora_config.target_modules} task={lora_config.task_type}")

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()
lora_model.config.id2label = {i: name for i, name in enumerate(label_encoder.classes_)}
lora_model.config.label2id = {name: i for i, name in enumerate(label_encoder.classes_)}
print(f"classifier head: {lora_model.classifier.out_proj.out_features} logits (== {NUM_LABELS} intents)")


In [ ]:
# 7.3 Trainer-based fine-tuning + save — RUN ON COLAB GPU (takes ~10-20 min on a T4)
import numpy as np
from sklearn.metrics import accuracy_score
from transformers import Trainer

def compute_metrics(eval_pred):
    """Accuracy on the validation split (drives load_best_model_at_end)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

roberta_trainer = Trainer(
    model=lora_model,
    args=roberta_args,
    train_dataset=train_roberta,
    eval_dataset=test_roberta,
    compute_metrics=compute_metrics,
)

train_result = roberta_trainer.train()
print(f"training done — global steps: {train_result.global_step} | train loss: {train_result.training_loss:.4f}")
print(roberta_trainer.evaluate())

# Save LoRA adapter weights + tokenizer for reuse (small files — the frozen backbone is re-downloaded via MODEL_NAME)
SAVE_DIR = "roberta-banking77-lora"
roberta_trainer.save_model(SAVE_DIR)
roberta_tokenizer.save_pretrained(SAVE_DIR)
print(f"saved LoRA adapter + tokenizer to ./{SAVE_DIR} (load later with AutoModelForSequenceClassification + PeftModel)")


In [ ]:
# 7.4 Evaluate on test_roberta_loader — RUN ON COLAB GPU (after 7.3)
# SAME metric format as the MLP eval (Section 5.3) so the baseline comparison is one-to-one.
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

lora_model.eval()
roberta_preds, roberta_true = [], []
with torch.no_grad():
    for batch in test_roberta_loader:
        input_ids = batch["input_ids"].to(lora_model.device)
        attention_mask = batch["attention_mask"].to(lora_model.device)
        logits = lora_model(input_ids=input_ids, attention_mask=attention_mask).logits
        roberta_preds += logits.argmax(dim=1).cpu().tolist()
        roberta_true += batch["labels"].tolist()

test_acc = accuracy_score(roberta_true, roberta_preds)
macro_f1 = f1_score(roberta_true, roberta_preds, average="macro")
weighted_f1 = f1_score(roberta_true, roberta_preds, average="weighted")
print(f"test accuracy: {test_acc:.4f} | macro F1: {macro_f1:.4f} | weighted F1: {weighted_f1:.4f}")

report = classification_report(
    roberta_true, roberta_preds, target_names=list(label_encoder.classes_), digits=3, output_dict=True
)
per_intent = pd.DataFrame(report).T.iloc[:-3]  # drop accuracy/macro/weighted rows
print(f"\nmacro avg: P={report['macro avg']['precision']:.3f} "
      f"R={report['macro avg']['recall']:.3f} F1={report['macro avg']['f1-score']:.3f}")
print(f"weighted avg: P={report['weighted avg']['precision']:.3f} "
      f"R={report['weighted avg']['recall']:.3f} F1={report['weighted avg']['f1-score']:.3f}")
print("\n--- Top-5 intents by F1 ---")
display(per_intent.sort_values("f1-score", ascending=False).head(5).round(3))
print("--- Bottom-5 intents by F1 ---")
display(per_intent.sort_values("f1-score").head(5).round(3))

# Confusable pairs: largest off-diagonal cells of the confusion matrix
cm = confusion_matrix(roberta_true, roberta_preds, labels=list(range(len(label_encoder.classes_))))
names = list(label_encoder.classes_)
pairs = [(cm[i, j], names[i], names[j])
         for i in range(len(names)) for j in range(len(names)) if i != j]
pairs.sort(reverse=True)
print("\n--- Top-5 confusable pairs (true -> predicted: count) ---")
for count, true_name, pred_name in pairs[:5]:
    print(f"  {true_name} -> {pred_name}: {count}")


**Comparing with the MLP baseline.** Read the Section 7.4 line (`test accuracy | macro F1 | weighted F1`) against the Section 5.3 line (MLP test acc 0.847): accuracy shows the overall win, macro-F1 shows whether rare intents improved, and the bottom-5 / confusable-pair lists show WHERE the transformer fixed (or still misses) the MLP's errors. Copy both lines into the final report table — no extra code needed.